<a href="https://colab.research.google.com/github/Alilson2/Projeto_IA/blob/main/GerarDataFrame.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import xarray as xr # Ler arquivos netcdf
from google.colab import drive
import seaborn as sns

import matplotlib.patches as mpatches # Desenhar geometria em um mapa

In [ ]:
import os
import xarray as xr

# --- Clonar repositório se não existir ---
if not os.path.exists("Projeto_IA"):
    !git clone https://github.com/Alilson2/Projeto_IA.git
else:
    print("📁 Repositório 'Projeto_IA' já existe — pulando o clone.")

# --- Verificar se a pasta foi criada ---
if not os.path.exists("Projeto_IA"):
    raise FileNotFoundError("❌ A pasta 'Projeto_IA' não foi encontrada. O clone pode ter falhado.")
else:
    print("\n✅ Repositório clonado com sucesso!\n")
    print("Arquivos dentro da pasta Projeto_IA:\n", os.listdir("Projeto_IA"))

# --- Localizar arquivos .nc ---
arquivos_nc = [f for f in os.listdir("Projeto_IA") if f.endswith(".nc")]
if not arquivos_nc:
    raise FileNotFoundError("❌ Nenhum arquivo .nc encontrado na pasta Projeto_IA!")
else:
    print("\n📂 Arquivo(s) NetCDF encontrado(s):")
    for f in arquivos_nc:
        print(" -", f)

# --- Montar lista de caminhos ---
ARQUIVO = [os.path.join("Projeto_IA", f) for f in arquivos_nc]

# --- Função para corrigir longitude ---
def corrigir_longitude(ds):
    for coord in ["longitude", "lon"]:
        if coord in ds.coords:
            ds = ds.assign_coords({coord: ((ds[coord] + 180) % 360) - 180})
            ds = ds.sortby(coord)
    return ds

# --- Abrir arquivos com segurança (nova sintaxe) ---
try:
    dados = xr.open_mfdataset(
        ARQUIVO,
        combine='by_coords',
        parallel=True,           # usa múltiplos núcleos
        preprocess=corrigir_longitude,
        combine_attrs='override' # 🟢 substitui o antigo compat='override'
    )
except ValueError as e:
    print("\n⚠️ Erro na combinação — tentando modo 'nested' (concat por tempo)...")
    dados = xr.open_mfdataset(
        ARQUIVO,
        combine='nested',
        concat_dim='valid_time',  # ajuste se sua dimensão temporal tiver outro nome
        parallel=True,
        preprocess=corrigir_longitude,
        combine_attrs='override'
    )

print("\n✅ Dataset carregado com sucesso!\n")

Cloning into 'Projeto_IA'...
remote: Enumerating objects: 251, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 251 (delta 40), reused 24 (delta 24), pack-reused 205 (from 1)
Receiving objects: 100% (251/251), 163.92 MiB | 33.08 MiB/s, done.
Resolving deltas: 100% (64/64), done.

✅ Repositório clonado com sucesso!

Arquivos dentro da pasta Projeto_IA:
 ['df_diario.csv', '(DEZ 2020-25 a 31).nc', 'MAR 2022 - 1a31.nc', '(MARÇO 2024 - 1a15) 27b35cf2be7d06ee99adbf9b6c8a7e3d.nc', '(ABR 2020-1 a 24).nc', 'ProjetoIA_ver_David.ipynb', '(DEZEMBRO 2023 - 1a15).nc', '(JAN 2024 - 1a15) 1ef5ab89e05ec50451a473eaac0197a5.nc', 'JUN 2022 - 16a30.nc', '(OUTUBRO 2024 - 16a31) b3f505f9374108f770a431d1f104a163.nc', 'FEV 2022 - 1a15.nc', '(DEZEMBRO 2023 - 16a31).nc', 'JUL 2022 - 16a31.nc', 'AGO 2022 - 16a31.nc', '(JULHO 2023 - 1a15) 6fdadb7142db0ca316d72f4c3cf60eec.nc', 'DEZ 2022 - 1a15.nc', '(MAR 2020-26 a 31).nc', '(FEV 2021-16 a 28).nc', '(

/tmp/ipython-input-2929652824.py:39: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'valid_time' ('valid_time',) The recommendation is to set join explicitly for this case.
  dados = xr.open_mfdataset(
/tmp/ipython-input-2929652824.py:39: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'latitude' ('latitude',) The recommendation is to set join explicitly for this case.
  dados = xr.open_mfdataset(



⚠️ Erro na combinação — tentando modo 'nested' (concat por tempo)...


/tmp/ipython-input-2929652824.py:48: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'latitude' ('latitude',) The recommendation is to set join explicitly for this case.
  dados = xr.open_mfdataset(
/tmp/ipython-input-2929652824.py:48: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'longitude' ('longitude',) The recommendation is to set join explicitly for this case.
  dados = xr.open_mfdataset(



✅ Dataset carregado com sucesso!



In [ ]:
df = dados.to_dataframe()
df = df.dropna()

# --- 2️⃣ Verificar se há a coordenada temporal ---
if "valid_time" not in dados.coords:
    raise ValueError("❌ O dataset não contém uma coordenada temporal chamada 'valid_time'.")

# --- 3️⃣ Converter o eixo temporal para pandas.DatetimeIndex ---
tempo = pd.to_datetime(dados["valid_time"].values)

# --- 4️⃣ Criar DataFrame com componentes temporais ---
df_tempo = pd.DataFrame({
    "timestamp": tempo,
    "timestamp_segundos": tempo.view("int64"),   # segundos desde 1970
    "ano": tempo.year,
    "mes": tempo.month,
    "dia": tempo.day,
    "hora": tempo.hour,
    "minuto": tempo.minute,
    "segundo": tempo.second,
    "dia_semana": tempo.dayofweek,
    "dia_do_ano": tempo.dayofyear
})

#print(df_tempo.head())

# Exemplo: seleciona uma variável e um período
# Sort the dataset by valid_time before slicing
dados_sorted = dados.sortby('valid_time')
dados_filtrado = dados_sorted

# Converte para pandas sem estourar RAM
df_panda = dados_filtrado.to_dataframe().reset_index()
df_panda = df_panda.dropna()

# Junta com df_tempo
df_final = pd.merge(
    df_panda,
    df_tempo,
    left_on='valid_time',
    right_on='timestamp',
    how='left'
)


df_final['longitude'] = np.round(df_final['longitude'], 1)
df_final['latitude'] = np.round(df_final['latitude'], 1)

In [ ]:
# ============================================================
# 1) Seleção de coordenadas
# ============================================================

def selecionar_coordenadas(df, N_lim, S_lim, L_lim, O_lim):

    df = df[
        (df["latitude"] <= N_lim) &
        (df["latitude"] >= S_lim) &
        (df["longitude"] >= O_lim) &
        (df["longitude"] <= L_lim)
    ]

    df = df.drop_duplicates(subset=["valid_time", "latitude", "longitude"])
    return df



# ============================================================
# 2) Cálculo de vapor, RH e VPD
# ============================================================

def calcula_vapor_umidade(df):

    t2m_c = df["t2m"] - 273.15
    d2m_c = df["d2m"] - 273.15

    es = 610.94 * np.exp(17.625 * t2m_c / (243.04 + t2m_c))
    ev = 610.94 * np.exp(17.625 * d2m_c / (243.04 + d2m_c))

    df["es"]  = es
    df["ev"]  = ev
    df["RH"]  = 100 * ev / es
    df["VPD"] = (es - ev) / 1000

    return df



# ============================================================
# 3) Agregar por dia e por pixel (SEM STD)
# ============================================================

def criar_dados(df):

    df["data"] = pd.to_datetime(df["valid_time"]).dt.date

    # Variáveis com estatísticas: mean, min, max
    vars_stats = ['d2m', 't2m', 'u10', 'v10', 'sp', 'es', 'ev', 'RH', 'VPD']

    # Variáveis que usam somente o último valor do dia
    vars_last = ["e", "ssrd", "sshf", "slhf", "tp"]

    g = df.groupby(["data", "latitude", "longitude"])

    resultados = {}

    # ---------- Estatísticas ----------
    for var in vars_stats:
        resultados[f"{var}_mean"] = g[var].mean()
        resultados[f"{var}_min"]  = g[var].min()
        resultados[f"{var}_max"]  = g[var].max()

    # ---------- Último valor ----------
    for var in vars_last:
        resultados[f"{var}_last"] = g[var].last()

    df_pixel = pd.concat(resultados, axis=1).reset_index()
    return df_pixel



# ============================================================
# 4) Agregar regionalmente (AQUI sim tem STD)
# ============================================================

def agregar_regional(df_pixel):

    df_pixel["data"] = pd.to_datetime(df_pixel["data"])
    df_pixel["mes"]  = df_pixel["data"].dt.month

    g = df_pixel.groupby("data")
    resultados = {}

    for col in df_pixel.columns:
        if col in ["data", "latitude", "longitude", "mes"]:
            continue

        # Últimos valores → média espacial
        if col.endswith("_last"):
            var = col.replace("_last", "")
            resultados[f"{var}_mean"] = g[col].mean()
            continue

        # Estatísticas (mean/min/max por pixel por dia)
        if "_mean" in col:
            var = col.replace("_mean", "")
            resultados[f"{var}_mean"] = g[col].mean()
            resultados[f"{var}_std"]  = g[col].std()
            continue

        if "_min" in col:
            var = col.replace("_min", "")
            resultados[f"{var}_min"]  = g[col].min()
            continue

        if "_max" in col:
            var = col.replace("_max", "")
            resultados[f"{var}_max"]  = g[col].max()
            continue

    df_regional = pd.concat(resultados, axis=1).reset_index()
    df_regional["mes"] = df_regional["data"].dt.month

    return df_regional



# ============================================================
# 5) Pipeline completo
# ============================================================

def pipeline(df_raw, N_lim, S_lim, L_lim, O_lim):

    df = selecionar_coordenadas(df_raw, N_lim, S_lim, L_lim, O_lim)
    df = calcula_vapor_umidade(df)
    df_daily_pixel = criar_dados(df)
    df_regional = agregar_regional(df_daily_pixel)

    return df_regional


In [ ]:
df_panda = df_final
# --- Aplicar ao DataFrame original ---
#resultados = calcula_vapor_umidade(df_panda)

# Adiciona novas colunas ao df_panda
#df_panda = pd.concat([df_panda.reset_index(drop=True), resultados.reset_index(drop=True)], axis=1)

O_lim = -46.84
L_lim = -46.36
N_lim = -23.4
S_lim = -23.74


df_diario = pipeline(df_panda, N_lim, S_lim, L_lim, O_lim)

In [ ]:
df_diario

,data,d2m_mean,d2m_std,d2m_min,d2m_max,t2m_mean,t2m_std,t2m_min,t2m_max,u10_mean,...,VPD_mean,VPD_std,VPD_min,VPD_max,e_last,ssrd_last,sshf_last,slhf_last,tp_last,mes
0,2020-01-01,291.728394,0.443505,287.502930,295.051270,298.340607,0.258781,292.407227,304.443420,0.847647,...,1.130807,0.039375,0.029082,2.871634,-0.005319,26790456.0,-2.698012e+06,-13302983.00,0.000216,1
1,2020-01-02,292.732941,0.441644,290.056641,294.872559,296.086304,0.194356,293.143677,300.906006,0.993034,...,0.532534,0.049364,0.045966,1.366565,-0.003221,13971246.0,-1.719732e+06,-8055768.00,0.024639,1
2,2020-01-03,291.953796,0.218227,290.480103,293.429810,293.977966,0.170310,292.171265,297.032471,-0.086623,...,0.293841,0.032776,0.018166,0.876933,-0.003729,16830144.0,-3.299828e+06,-9325415.00,0.004963,1
3,2020-01-04,291.532410,0.229106,290.265747,293.497375,293.635315,0.171876,290.890747,297.959717,-0.269839,...,0.306237,0.031990,0.035693,0.906694,-0.003343,16385261.0,-2.567596e+06,-8360272.00,0.018063,1
4,2020-01-05,291.924286,0.153417,290.449219,293.561707,293.963013,0.120610,290.509766,298.421570,-0.627977,...,0.313766,0.016571,0.005157,0.958426,-0.003879,20081940.0,-3.342363e+06,-9700072.00,0.015074,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1822,2024-12-27,293.267975,0.260209,292.221802,294.783081,294.402771,0.264179,293.053223,296.234741,1.078462,...,0.171804,0.017414,0.012472,0.458858,-0.001629,5370055.5,-1.591558e+04,-4074817.25,0.012803,12
1823,2024-12-28,292.958496,0.393401,291.610107,295.629761,295.519104,0.125146,292.610596,300.126099,0.095428,...,0.412606,0.056141,0.044639,1.285405,-0.004262,22325614.0,-3.248643e+06,-10659076.00,0.007477,12
1824,2024-12-29,292.292816,0.295455,290.683594,294.784180,294.359283,0.118079,290.832153,299.705566,-1.526076,...,0.321508,0.039310,0.005187,1.268487,-0.004134,22780396.0,-4.280936e+06,-10338085.00,0.005537,12
1825,2024-12-30,291.707825,0.221028,290.719849,293.534546,293.796265,0.147480,291.191406,298.225037,-1.339824,...,0.309567,0.037203,0.017640,0.989121,-0.003755,20647160.0,-3.574952e+06,-9389764.00,0.008517,12
